In [4]:
# 1. Install dependencies + accelerate
!pip install -q streamlit sentence-transformers transformers tokenizers pandas numpy accelerate
# 2. Install localtunnel
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [17]:
# 1. Clean up any existing directory if running again
!rm -rf economic-news-sentiment

# 2. Clone your clean GitHub repo (replace with your personal GitHub URL)
!git clone https://github.com/Nas-Mohd/economic-news-sentiment.git

# 3. CRITICAL: Move your notebook's working directory into the directory containing app.py
# Use '%cd' (IPython magic) instead of '!cd' (subshell) to keep the working directory active!
%cd economic-news-sentiment/demo

Cloning into 'economic-news-sentiment'...
remote: Enumerating objects: 644, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (193/193), done.
remote: Total 644 (delta 150), reused 158 (delta 68), pack-reused 383 (from 1)
Receiving objects: 100% (644/644), 8.79 MiB | 10.61 MiB/s, done.
Resolving deltas: 100% (393/393), done.
/content/economic-news-sentiment/demo/economic-news-sentiment/demo


In [5]:
print("Pre-downloading Hugging Face models to cache (this prevents timeouts)...")
from transformers import pipeline
from sentence_transformers import SentenceTransformer

# 1. Cache Semantic Router
print("\n[1/3] Caching all-MiniLM-L6-v2...")
_ = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. Cache Fine-Tuned DeBERTa (Requires 'accelerate' package)
print("\n[2/3] Caching Fine-Tuned DeBERTa Classifier (takes ~1 minute)...")
_ = pipeline(
    "text-classification",
    model="dummfak/deberta-v3-base-macroeconomic-aspect-classifier",
    device_map="auto"
)

# 3. Cache Fine-Tuned ABSA FinBERT
print("\n[3/3] Caching Fine-Tuned ABSA FinBERT...")
_ = pipeline(
    "text-classification",
    model="dummfak/finbert-macroeconomic-absa"
)

print("\n🎉 All models cached successfully and ready!")

Pre-downloading Hugging Face models to cache (this prevents timeouts)...

[1/3] Caching all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]


🎉 All models cached successfully and ready!


In [21]:
# 3. Download the standalone Cloudflare Tunnel client daemon
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared-linux-amd64

In [22]:
# 4. Launch Streamlit in the background, start the Cloudflare Tunnel, and output your direct link
import subprocess
import time
import re

print("🚀 Launching Streamlit in the background...")
# Run Streamlit on port 8501, redirecting output to streamlit.log
with open("streamlit.log", "w") as f_st:
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "127.0.0.1"], stdout=f_st, stderr=f_st)

# Wait a brief moment for Streamlit server to spin up
time.sleep(3)

print("🌐 Starting Cloudflare Tunnel...")
# Start free Cloudflare Tunnel proxying port 8501
with open("tunnel.log", "w") as f_tun:
    subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8501"], stdout=f_tun, stderr=f_tun)

# Allow the tunnel to perform handshake and receive the URL
time.sleep(30)

# Extract the trycloudflare URL automatically from your tunnel.log file
tunnel_url = None
with open("tunnel.log", "r") as f_log:
    log_data = f_log.read()
    matches = re.findall(r'https?://[a-zA-Z0-9-]+\.trycloudflare\.com', log_data)
    if matches:
        tunnel_url = matches[0]

if tunnel_url:
    print("\n" + "="*65)
    print("✨ SUCCESS: Your Economic News Sentiment ABSA App is operational!")
    print(f"🔗 Click this official link: {tunnel_url}")
    print("="*65)
else:
    print("\n⚠️ Tunnel URL extraction timed out.")
    print("Run the command below to locate your generated link:")
    !cat tunnel.log

🚀 Launching Streamlit in the background...
🌐 Starting Cloudflare Tunnel...

✨ SUCCESS: Your Economic News Sentiment ABSA App is operational!
🔗 Click this official link: https://circuit-destinations-roommates-navigator.trycloudflare.com


In [11]:
import urllib

# Print the bypass IP address
try:
    ip_addr = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("👉 Your Tunnel Password / IP is:", ip_addr)
    print("Copy this IP into the 'Tunnel Password' field when the localtunnel page opens.\n")
except Exception as e:
    print("Could not retrieve IP address:", e)

# Run localtunnel
!npx localtunnel --port 8501

👉 Your Tunnel Password / IP is: 35.198.221.193
Copy this IP into the 'Tunnel Password' field when the localtunnel page opens.

⠙⠹⠸⠼your url is: https://true-points-flash.loca.lt
^C
